In [141]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, input_guardrail, output_guardrail, GuardrailFunctionOutput
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio



In [142]:
load_dotenv(override=True)

True

In [143]:
hook_instructions = "you are a professional script writer specializing in writing hooks for instagram reels/youtube shorts" \
"You have to give the best 3 hooks for a topic which can make the end user watch/click the video"

script_instructions = "you are a professional script writer specializing in writing scripts for instagram reels/youtube shorts" \
"You have to give the best script for a topic which will make the end user watch the video. The scripts" \
"should be concise, ENGAGING, INTERESTING, HOOKING and SHOULD BE ABLE TO BE COVERED VERBALLY IN 40 SECONDS."

hook_agent = Agent(name="hook_agent", instructions=hook_instructions, model='gpt-4o-mini')
hook_tool = hook_agent.as_tool(tool_name='hook_tool', tool_description='Create an engaging, interesting, viral hook for the topic given')

script_agent = Agent(name="script_agent", instructions=script_instructions, model='gpt-4o-mini')
script_tool = script_agent.as_tool(tool_name='script_agent',tool_description='Write an engaging, hooking and interesting script for a given topic')

In [144]:
tools = [hook_tool, script_tool]

In [145]:
#sending email
@function_tool
def send_email(body: str):
    """ USE THIS TOOL TO Send out an email with the given body to the influencer """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("aarpitbharti007@gmail.com")  # Change to your verified sender
    to_email = To("arpit.bharti8503@gmail.com")  # Change to your recipient
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Script email", content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [146]:
#formatting agent

format_agent_instructions = """
You are an expert Script Editor. Your job is to take the provided hook and script and format them into a clean, professional HTML email for an influencer.

Follow these formatting rules:
1. Use <h2> tags for section headers (e.g., 'THE HOOKS', 'THE SCRIPT').
2. Wrap the Hooks in an unordered list (<ul>) with <strong> tags for emphasis.
3. For the Script: 
   - Break it into short, punchy paragraphs.
   - Use <br> for line breaks to indicate pauses.
   - Use <b>[BOLD CAPS]</b> for visual cues or on-screen text instructions.
4. Keep the original wording, but refine the layout to make it 'scannable' for a fast-paced recording session.
5. Once the HTML body is ready, call the send_email tool immediately.
"""

format_agent = Agent(name='format_agent',
                     instructions=format_agent_instructions,
                    model='gpt-4o-mini',
                    tools=[send_email],
                    handoff_description='Format a script and send it out as an email')


In [147]:
tools = [hook_tool, script_tool]
handoffs1 = [format_agent]


In [148]:
from pydantic import BaseModel
#guardRailing
class input_gr(BaseModel):
    is_sensitive : bool
    topic : str

input_guard_agent = "You are a input guardrailing agent. you have to check if there is any sensitive" \
"topic like politics/human trafficiking/suicides something similar is passed as an input."

guard_agent = Agent(name="input_sensitive_agent",
                    instructions=input_guard_agent,
                    output_type=input_gr,
                    model='gpt-4o-mini')


In [158]:
class output_gr(BaseModel):
    is_rejected : bool
    content : str
    
out_agent = Agent(name="out_agent",
                  instructions="Check if the final script is more than 50 words",
                  output_type=output_gr,
                  model='gpt-4o-mini')

@output_guardrail
async def output_gr(ctx, message, agent):
    result = await Runner.run(out_agent, message, context=ctx.context)
    rejected = result.final_output.is_rejected
    return GuardrailFunctionOutput(output_info={'content':result.final_output}, tripwire_triggered=rejected)



In [155]:
@input_guardrail
async def sensitive_gr(ctx, agent, message):
    result = await Runner.run(guard_agent, message, context=ctx.context)
    is_sensitive = result.final_output.is_sensitive
    return GuardrailFunctionOutput(output_info={'sensitive_topic' :result.final_output}, tripwire_triggered=is_sensitive)


In [ ]:
#boss

main_ins = "You are an expert script writer. Your job is to: " \
"1. Use hook_tool to get hooks for the topic. " \
"2. Use script_tool to get the script. " \
"3. Use critic tool to review and if it not approved, use the hook_tool and script_tool to write them again" \
"with the feedback incorporated. Use this maximum two times." \
"3. Immediately hand off to the format_agent. " \
"Do not loop or repeat steps."

main_agent = Agent(name="main_agent",
                   instructions=main_ins,
                   tools=tools,
                   handoffs=handoffs1,
                   input_guardrails=[sensitive_gr, output_gr],
                   model='gpt-4o-mini')


In [ ]:
print(hook_tool)
print(script_tool)
print(send_email)
print(format_agent)

FunctionTool(name='hook_tool', description='Create an engaging, interesting, viral hook for the topic given', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'hook_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000261E06E4C20>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)
FunctionTool(name='script_agent', description='Write an engaging, hooking and interesting script for a given topic', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'script_agent_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000261E06E4EA0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=Non

In [157]:
message = "write me a script on Future of BJP in India"
with trace("influencer agent with critic"):
    result = await Runner.run(main_agent, message)
    print(result.final_output)

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire